# Credit Risk Profiling & NPA Monitoring

End-to-end portfolio analysis using a synthetic retail-loan dataset. The notebook covers EDA, feature engineering, logistic-regression PD modelling, risk segmentation and model evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve

df = pd.read_csv('../data/loan_portfolio.csv', parse_dates=['application_date'])
df.head()


## 1. Portfolio EDA


In [ ]:
print(df.shape)
print(df.isna().sum().sort_values(ascending=False).head())
print(df[['loan_amount','credit_score','dti_pct','ltv_pct','pd_estimate','risk_score']].describe().T)


In [ ]:
monthly = df.groupby(df['application_date'].dt.to_period('M')).agg(
    loans=('loan_id','count'), exposure=('loan_amount','sum'),
    npa_ratio=('npa_flag','mean'), avg_pd=('pd_estimate','mean')
).reset_index()
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(monthly['application_date'].astype(str), monthly['npa_ratio']*100)
ax.set_title('Monthly NPA Ratio'); ax.set_ylabel('NPA %'); ax.tick_params(axis='x', rotation=90)
plt.tight_layout()


## 2. Feature engineering and train/test split


In [ ]:
target = 'default_flag'
features = ['age','annual_income','employment_years','loan_amount','tenure_months','interest_rate_pct',
            'credit_score','dti_pct','prior_delinquencies','ltv_pct','dependents','purpose',
            'employment_type','region','existing_loans','monthly_obligations']
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
pre = ColumnTransformer([('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_cols),
                         ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]), cat_cols)])
model = Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1000, class_weight='balanced'))])
model.fit(X_train, y_train)


## 3. Model evaluation


In [ ]:
proba = model.predict_proba(X_test)[:,1]
pred = (proba >= 0.20).astype(int)  # lower threshold to prioritize risk capture
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))
print(classification_report(y_test, pred, digits=3))


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(6,5)); plt.plot(fpr,tpr,label=f'AUC={roc_auc_score(y_test,proba):.3f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve'); plt.legend(); plt.tight_layout()


## 4. Risk bands and monitoring logic

Operational bands: Low (<5% PD), Moderate (5–10%), High (10–20%), Very High (>20%). Early-warning status is triggered by 30–89 DPD, high DTI, high LTV, repeated delinquency or low bureau score. These thresholds are illustrative and should be recalibrated using a bank's historical outcomes and policy.


In [ ]:
def band(p):
    if p < .05: return 'Low'
    if p < .10: return 'Moderate'
    if p < .20: return 'High'
    return 'Very High'
df['model_pd'] = model.predict_proba(df[features])[:,1]
df['model_risk_band'] = df['model_pd'].map(band)
df['early_warning'] = ((df.days_past_due.between(30,89)) | (df.dti_pct>=50) | (df.ltv_pct>=85) | (df.prior_delinquencies>=2) | (df.credit_score<650)).astype(int)
df[['loan_id','model_pd','model_risk_band','early_warning']].head()


## 5. Export model scoring output


In [ ]:
out = df[['loan_id','application_date','loan_amount','credit_score','dti_pct','ltv_pct','days_past_due','npa_flag','model_pd','model_risk_band','early_warning']]
out.to_csv('../outputs/model_scoring_output.csv', index=False)
